# GeoGuess-HAI: Colab pipeline

This notebook runs the full Assignment 2 pipeline on a **T4 GPU**: dataset → dev predictions + temperature fit →
JSON baseline → study predictions → evaluation → live server. Every step calls the same CLI as `README.md`.

Before you start, select **Runtime → Change runtime type → T4 GPU**. Run the cells top to bottom. `predict.py`
appends to its JSONL after every trial, so if the session times out, re-run the setup cells and then the
step that was interrupted. It resumes where it stopped.

## 1. GPU check

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - switch the runtime to T4")

## 2. Get the repo

Option A clones from git; set `REPO_URL` first. Option B uploads a zip of the repo (`geoguess-hai.zip`, with the
repo files at the top level or in one folder). If you are uploading pre-built `data/` and `outputs/`, include
them in the zip.

In [ ]:
import os, glob, zipfile
REPO_URL = "https://github.com/<your-user>/<your-repo>.git"   # placeholder: set this, or leave it to upload a zip
REPO_DIR = "/content/geoguess-hai"

if not os.path.exists(REPO_DIR):
    if "<your-user>" not in REPO_URL:
        !git clone -q {REPO_URL} {REPO_DIR}
    else:
        from google.colab import files
        up = files.upload()                      # choose the repo zip
        zipfile.ZipFile(next(iter(up))).extractall("/content/_repo")
        root = os.path.dirname(glob.glob("/content/_repo/**/config.yaml", recursive=True)[0])
        os.rename(root, REPO_DIR)
%cd {REPO_DIR}
!ls

## 3. Install

Colab already has `torch`, so this mostly adds `transformers`, `easyocr`, `fastapi`, `streamlit` and the rest
of `requirements.txt`.

In [ ]:
!pip install -q -r requirements.txt

## 4. Hugging Face cache and model download

The model is `Qwen/Qwen3-VL-2B-Instruct`, pinned to the revision in `config.yaml` (about 4 GB, no token needed).
Set `USE_DRIVE = True` to keep the cache on Google Drive, so a new session does not download it again.
`os.environ` is inherited by the later `!python ...` commands.

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"

import yaml
from huggingface_hub import snapshot_download
m = yaml.safe_load(open("config.yaml"))["model"]
print(snapshot_download(m["model_id"], revision=m["revision"]))

## 5. Dataset (SPEC 4)

**Option A, build here.** This downloads OSV-5M `test.csv` and only the images that are needed, then filters,
dedups, runs OCR (GPU) and writes `data/{study,dev}/` plus a contact sheet per split.

**Option B, upload a pre-built `data/`.** Use this if the dataset and manual screen were done elsewhere. Upload
`data.zip`, which contains `data/study`, `data/dev`, `data/screen_*.txt`, and `data/countries.txt` if it exists.

The **manual screen** is required (README step 2). Download `data/study/contact_sheet.html` and
`data/dev/contact_sheet.html` with their images, tick the boxes, and put the generated lists into
`data/screen_exclude.txt` and `data/screen_text.txt`. Then re-run Option A, which is fast because downloads and
OCR are cached.

In [ ]:
BUILD = True
if BUILD:
    !python src/build_dataset.py --config config.yaml --contact-sheet
else:
    from google.colab import files
    up = files.upload()                          # data.zip with a top-level data/ folder
    zipfile.ZipFile(next(iter(up))).extractall(".")
!tail -n 30 data/build_log.txt
!wc -l data/study/study_meta.csv data/dev/dev_meta.csv

In [ ]:
# Optional: zip both contact sheets with their images, to screen them on your laptop
!cd data && zip -qr ../contact_sheets.zip study/contact_sheet.html study/*.jpg dev/contact_sheet.html dev/*.jpg
from google.colab import files
files.download("contact_sheets.zip")

## 6. Dev predictions + temperature fit (SPEC 5.2, milestone 3)

This writes `outputs/dev/raw_outputs.jsonl`, `outputs/dev/dev_dataset.csv` and `outputs/temperature.json`.
For a quick smoke test, add `--limit 5` first.

In [ ]:
!python src/predict.py --split dev --config config.yaml --fit-temperature
!cat outputs/temperature.json

Dev evaluation, including the comparison of length-normalized and summed scoring. Its report goes to a
separate folder, so the study report is not overwritten. Record the numbers in `docs/iteration_log.md`.

In [ ]:
!python src/evaluate.py --pred outputs/dev/dev_dataset.csv --raw outputs/dev/raw_outputs.jsonl --out outputs/dev/report

## 7. JSON baseline (SPEC 5.4, iteration log only)

This reports the parse-failure rate and accuracy of free-form JSON answers in
`outputs/dev/baseline_json_summary.json`.

In [ ]:
!python src/predict.py --split dev --config config.yaml --baseline-json

## 8. Study predictions (milestone 6)

Run this only once the dev metrics are within the SPEC 0.3 targets. It writes `outputs/raw_outputs.jsonl` and
`outputs/study_dataset.csv`, the submitted dataset. Take a screenshot of the progress bar for the report.

In [ ]:
!python src/predict.py --split study --config config.yaml

## 9. Evaluate (SPEC 7, 8.3)

Annotation (`streamlit run src/annotate.py ...`) is easiest on a laptop. Upload `outputs/annotations.csv` (and
`outputs/annotations_rater2.csv` if there is one) before running this cell. Without them, the error table is
left as "annotations not provided".

In [ ]:
args = "--pred outputs/study_dataset.csv --raw outputs/raw_outputs.jsonl"
if os.path.exists("outputs/annotations.csv"):
    args += " --ann outputs/annotations.csv"
if os.path.exists("outputs/annotations_rater2.csv"):
    args += " --ann2 outputs/annotations_rater2.csv"
!python src/evaluate.py {args}

## 10. Figures and metrics

In [ ]:
import json
from IPython.display import Image, Markdown, display
display(Markdown(open("outputs/report/summary.md").read()))
for f in ["reliability.png", "confusion_matrix.png", "per_country_accuracy.png"]:
    display(Image(f"outputs/report/{f}", width=700))
mt = json.load(open("outputs/report/metrics.json"))
print({k: mt[k] for k in ["n", "top1", "top3", "mrr", "ece"]})

## 11. Live inference server (SPEC 10)

This starts `uvicorn src.server:app` in the background, waits for the model to load, then calls `/health` and
`/predict`. Screenshot the `/predict` output as the "evidence of model running". It loads its own copy of the
model, which fits on a T4 because the `predict.py` processes above have exited.

In [ ]:
import subprocess, time, urllib.request
server = subprocess.Popen("uvicorn src.server:app --host 0.0.0.0 --port 8000 > server.log 2>&1", shell=True)
for _ in range(120):                              # model load takes ~1 min
    try:
        print(urllib.request.urlopen("http://localhost:8000/health").read().decode()); break
    except Exception:
        time.sleep(5)
else:
    print(open("server.log").read()[-3000:])

In [ ]:
import pandas as pd
from IPython.display import Image, display
tid = pd.read_csv("data/study/study_meta.csv")["trial_id"].iloc[0]
!curl -s -X POST localhost:8000/predict -H 'Content-Type: application/json' -d '{{"trial_id": "{tid}"}}' | python -m json.tool
display(Image(f"data/study/{tid}.jpg", width=500))

### Optional: public tunnel

This exposes the server through a cloudflared quick tunnel, which needs no account. The study app can then use
it live: set `GEOHAI_SERVER_URL=<url>` and open the app with `?live=1`. ngrok also works, but it needs an auth
token.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import re
tunnel = subprocess.Popen("./cloudflared tunnel --url http://localhost:8000 > tunnel.log 2>&1", shell=True)
for _ in range(30):
    time.sleep(2)
    hit = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if hit:
        print("public URL:", hit.group(0), "| try:", hit.group(0) + "/health"); break

## 12. Screenshot checklist (SPEC 13)

Save these to `docs/screenshots/`:
- Colab model loading: the output of the cell in section 4 or 6, showing the device and the model ID.
- `predict.py` progress: the tqdm bar from section 8.
- A `/predict` response: the JSON from section 11.
- One rendered trial: run `streamlit run study_app/app.py` locally and capture a Human-AI trial screen.

Then download the outputs:

In [ ]:
from google.colab import files
!zip -qr outputs.zip outputs docs/iteration_log.md
files.download("outputs.zip")